# IMF WEO Dataframe Demo

This notebook shows the recommended analysis workflow using `pysdmx` and pandas against the IMF WEO SDMX service.

In [1]:
from io import StringIO

import pandas as pd
from pysdmx.api.dc.query import Operator, TextFilter
from pysdmx.api.qb import ApiVersion, DataContext, DataFormat, DataQuery, RestService

service = RestService(
    'https://api.imf.org/external/sdmx/3.0',
    ApiVersion.V2_0_0,
    data_format=DataFormat.SDMX_CSV_2_1_0,
    timeout=60,
)

In [2]:
# Example: United Kingdom, GDP current prices, U.S. dollars, billions.
query = DataQuery(
    context=DataContext.DATAFLOW,
    agency_id='IMF.RES',
    resource_id='WEO',
    version='+',
    key='GBR.NGDPD.A',
    components=TextFilter(field='SCALE', operator=Operator.IN, value=['9']),
    obs_dimension='TIME_PERIOD',
    attributes='dsd',
)
raw_csv = service.data(query).decode('utf-8')
raw_csv.splitlines()[:3]

Invalid: Client error 400: The query returned a 400 error code. The query was `https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.RES/WEO/+/GBR.NGDPD.A?c[SCALE]=9&dimensionAtObservation=TIME_PERIOD`. The error message was: `<!doctype html><html lang="en"><head><title>HTTP Status 400 – Bad Request</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 400 – Bad Request</h1></body></html>`.

In [3]:
df = pd.read_csv(StringIO(raw_csv), keep_default_na=False)
df.columns = [column.replace('[;]', '').strip() for column in df.columns]
df['TIME_PERIOD'] = df['TIME_PERIOD'].astype(int)
df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
df = df.rename(
    columns={
        'COUNTRY': 'country_code',
        'INDICATOR': 'indicator_code',
        'TIME_PERIOD': 'time_period',
        'OBS_VALUE': 'obs_value',
        'COUNTRY_UPDATE_DATE': 'country_update_date',
    }
)
df[['country_code', 'indicator_code', 'time_period', 'obs_value', 'SCALE', 'country_update_date']].tail(10)

NameError: name 'raw_csv' is not defined

Use the shared helpers in `weo_tools` when you want the legacy label compatibility and Excel shaping logic. Use the direct `pysdmx` pattern above when you want a small, explicit notebook workflow.